In [1]:
import numpy as np
import pandas as pd
import gc

In [2]:
def downcast_dtypes(df):
    cols = df.dtypes.index.tolist()
    types = df.dtypes.values.tolist()
    for i, t in enumerate(types):
        if 'int' in str(t):
            if df[cols[i]].min() > np.iinfo(np.int8).min and df[cols[i]].max() < np.iinfo(np.int8).max:
                df[cols[i]] = df[cols[i]].astype(np.int8)
            elif df[cols[i]].min() > np.iinfo(np.int16).min and df[cols[i]].max() < np.iinfo(np.int16).max:
                df[cols[i]] = df[cols[i]].astype(np.int16)
            elif df[cols[i]].min() > np.iinfo(np.int32).min and df[cols[i]].max() < np.iinfo(np.int32).max:
                df[cols[i]] = df[cols[i]].astype(np.int32)
        elif 'float' in str(t):
            df[cols[i]] = df[cols[i]].astype(np.float32)
        elif t == object:
            if cols[i] != 'date':
                df[cols[i]] = df[cols[i]].astype('category')
    return df

In [3]:
# 1. Load datasets (Adjust path to your local folder)
DATA_DIR = "./m5-forecasting-accuracy/"
calendar = downcast_dtypes(pd.read_csv(f"{DATA_DIR}calendar.csv"))
prices = downcast_dtypes(pd.read_csv(f"{DATA_DIR}sell_prices.csv"))
sales = pd.read_csv(f"{DATA_DIR}sales_train_evaluation.csv")

In [4]:
calendar

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,29-01-2011,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,30-01-2011,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,31-01-2011,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,01-02-2011,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,02-02-2011,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1964,15-06-2016,11620,Wednesday,5,6,2016,d_1965,NaN,NaN,NaN,NaN,0,1,1
1965,16-06-2016,11620,Thursday,6,6,2016,d_1966,NaN,NaN,NaN,NaN,0,0,0
1966,17-06-2016,11620,Friday,7,6,2016,d_1967,NaN,NaN,NaN,NaN,0,0,0
1967,18-06-2016,11621,Saturday,1,6,2016,d_1968,NaN,NaN,NaN,NaN,0,0,0


In [6]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   date          1969 non-null   str  
 1   wm_yr_wk      1969 non-null   int16
 2   weekday       1969 non-null   str  
 3   wday          1969 non-null   int8 
 4   month         1969 non-null   int8 
 5   year          1969 non-null   int16
 6   d             1969 non-null   str  
 7   event_name_1  162 non-null    str  
 8   event_type_1  162 non-null    str  
 9   event_name_2  5 non-null      str  
 10  event_type_2  5 non-null      str  
 11  snap_CA       1969 non-null   int8 
 12  snap_TX       1969 non-null   int8 
 13  snap_WI       1969 non-null   int8 
dtypes: int16(2), int8(5), str(7)
memory usage: 172.7 KB


In [10]:
prices

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
...,...,...,...,...
6841116,WI_3,FOODS_3_827,11617,1.00
6841117,WI_3,FOODS_3_827,11618,1.00
6841118,WI_3,FOODS_3_827,11619,1.00
6841119,WI_3,FOODS_3_827,11620,1.00


In [7]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int16  
 3   sell_price  float32
dtypes: float32(1), int16(1), str(2)
memory usage: 252.9 MB


In [9]:
sales

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,0,0,2,2,...,1,0,3,0,1,1,0,0,1,1
30486,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
30487,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,0,6,0,2,...,0,0,1,2,0,1,0,1,0,2
30488,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,1,1,1,4,6,0,1,1,1,0


In [8]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), str(6)
memory usage: 454.7 MB


In [11]:
# 2. Scope for local execution: Store CA_1, Category FOODS, Days d_1200 to d_1941
sales = sales[(sales['store_id'] == 'CA_1') & (sales['cat_id'] == 'FOODS')]
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [f"d_{i}" for i in range(1200, 1942)]
sales = sales[id_cols + day_cols]

In [16]:
# 3. Melt from Wide to Long format
grid_df = pd.melt(sales, id_vars=id_cols, value_vars=day_cols, var_name='d', value_name='sales')
del sales
gc.collect()

0

In [17]:
grid_df

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1200,1
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1200,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1200,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1200,5
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1200,0
...,...,...,...,...,...,...,...,...
1066249,FOODS_3_823_CA_1_evaluation,FOODS_3_823,FOODS_3,FOODS,CA_1,CA,d_1941,2
1066250,FOODS_3_824_CA_1_evaluation,FOODS_3_824,FOODS_3,FOODS,CA_1,CA,d_1941,0
1066251,FOODS_3_825_CA_1_evaluation,FOODS_3_825,FOODS_3,FOODS,CA_1,CA,d_1941,1
1066252,FOODS_3_826_CA_1_evaluation,FOODS_3_826,FOODS_3,FOODS,CA_1,CA,d_1941,1


In [18]:
grid_df.describe()

,sales
count,1.066254e+06
mean,2.058045e+00
std,4.854528e+00
min,0.000000e+00
25%,0.000000e+00
50%,1.000000e+00
75%,2.000000e+00
max,2.600000e+02


In [19]:
# 4. Merge Calendar & Sell Prices
grid_df = grid_df.merge(calendar[['d', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 
                                  'event_name_1', 'snap_CA']], on='d', how='left')
grid_df = grid_df.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
grid_df = downcast_dtypes(grid_df)
grid_df['d_num'] = grid_df['d'].apply(lambda x: int(x.split('_')[1])).astype(np.int16)
print(f"Final shape: {grid_df.shape}, Memory: {grid_df.memory_usage().sum() / 1e6:.2f} MB")

Final shape: (1066254, 17), Memory: 171.70 MB


In [20]:
grid_df

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,wday,month,year,event_name_1,snap_CA,sell_price,d_num
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1200,1,12-05-2014,11415,3,5,2014,NaN,0,2.24,1200
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1200,0,12-05-2014,11415,3,5,2014,NaN,0,8.88,1200
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1200,0,12-05-2014,11415,3,5,2014,NaN,0,2.88,1200
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1200,5,12-05-2014,11415,3,5,2014,NaN,0,1.78,1200
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1200,0,12-05-2014,11415,3,5,2014,NaN,0,3.28,1200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066249,FOODS_3_823_CA_1_evaluation,FOODS_3_823,FOODS_3,FOODS,CA_1,CA,d_1941,2,22-05-2016,11617,2,5,2016,NaN,0,2.98,1941
1066250,FOODS_3_824_CA_1_evaluation,FOODS_3_824,FOODS_3,FOODS,CA_1,CA,d_1941,0,22-05-2016,11617,2,5,2016,NaN,0,2.48,1941
1066251,FOODS_3_825_CA_1_evaluation,FOODS_3_825,FOODS_3,FOODS,CA_1,CA,d_1941,1,22-05-2016,11617,2,5,2016,NaN,0,3.98,1941
1066252,FOODS_3_826_CA_1_evaluation,FOODS_3_826,FOODS_3,FOODS,CA_1,CA,d_1941,1,22-05-2016,11617,2,5,2016,NaN,0,1.28,1941


In [21]:
# Sort chronologically per item
grid_df = grid_df.sort_values(['id', 'd_num']).reset_index(drop=True)

# 1. Lag Features (Shifted by 28+ days to prevent data leakage across the 28-day horizon)
for lag in [28, 35, 42, 56]:
    grid_df[f'sales_lag_{lag}'] = grid_df.groupby('id', observed=False)['sales'].shift(lag).astype(np.float32)

# 2. Rolling Demand Statistics (Shifted by 28 days)
for win in [7, 14, 28, 56]:
    shifted = grid_df.groupby('id', observed=False)['sales'].shift(28)
    grid_df[f'roll_mean_28_{win}'] = shifted.transform(lambda x: x.rolling(win).mean()).astype(np.float32)
    grid_df[f'roll_std_28_{win}'] = shifted.transform(lambda x: x.rolling(win).std()).astype(np.float32)

# 3. Price Relative & Promotional Momentum Features
grid_df['price_max'] = grid_df.groupby(['store_id', 'item_id'], observed=False)['sell_price'].transform('max')
grid_df['price_norm'] = (grid_df['sell_price'] / grid_df['price_max']).astype(np.float32)
grid_df['price_momentum'] = (grid_df['sell_price'] / grid_df.groupby(['id'], observed=False)['sell_price'].shift(1)).astype(np.float32)

# Drop initial rows with NaN lags
grid_df = grid_df.dropna(subset=['sales_lag_56', 'sell_price']).reset_index(drop=True)

In [22]:
grid_df

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,roll_std_28_7,roll_mean_28_14,roll_std_28_14,roll_mean_28_28,roll_std_28_28,roll_mean_28_56,roll_std_28_56,price_max,price_norm,price_momentum
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1256,1,07-07-2014,11423,...,0.951190,0.928571,1.384768,0.857143,1.208436,NaN,NaN,2.24,1.0,1.0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1257,0,08-07-2014,11423,...,0.951190,0.928571,1.384768,0.785714,1.197440,NaN,NaN,2.24,1.0,1.0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1258,1,09-07-2014,11423,...,0.951190,0.928571,1.384768,0.785714,1.197440,NaN,NaN,2.24,1.0,1.0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1259,0,10-07-2014,11423,...,0.951190,0.571429,0.755929,0.785714,1.197440,NaN,NaN,2.24,1.0,1.0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1260,0,11-07-2014,11423,...,0.786796,0.571429,0.755929,0.821429,1.188013,NaN,NaN,2.24,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972370,FOODS_3_827_CA_1_evaluation,FOODS_3_827,FOODS_3,FOODS,CA_1,CA,d_1937,5,18-05-2016,11616,...,6.626067,6.714286,5.075258,5.035714,4.194195,4.589286,3.788868,1.00,1.0,1.0
972371,FOODS_3_827_CA_1_evaluation,FOODS_3_827,FOODS_3,FOODS,CA_1,CA,d_1938,7,19-05-2016,11616,...,6.855655,6.357143,5.153149,5.035714,4.194195,4.607143,3.778786,1.00,1.0,1.0
972372,FOODS_3_827_CA_1_evaluation,FOODS_3_827,FOODS_3,FOODS,CA_1,CA,d_1939,2,20-05-2016,11616,...,6.855655,6.500000,5.080127,5.071429,4.189303,4.642857,3.772766,1.00,1.0,1.0
972373,FOODS_3_827_CA_1_evaluation,FOODS_3_827,FOODS_3,FOODS,CA_1,CA,d_1940,10,21-05-2016,11617,...,6.993194,6.500000,5.080127,4.964286,4.203016,4.696429,3.726659,1.00,1.0,1.0
